In [9]:
import sounddevice as sd
import soundfile as sf
import numpy as np
import librosa
import tensorflow as tf

def record_audio(filename="recorded_10s.wav", duration=10, sr=16000):
    print("🎙️ Recording 10 seconds...")
    audio = sd.rec(int(duration * sr), samplerate=sr, channels=1, dtype='float32')
    sd.wait()
    sf.write(filename, audio, sr)
    print(f"✅ Saved recording as {filename}")
    return filename


In [10]:
class ElephantDetector:
    def __init__(self, model_path, sr=16000, chunk_duration=3):
        self.sr = sr
        self.chunk_duration = chunk_duration
        self.chunk_samples = sr * chunk_duration

        # Load TFLite Model
        self.interpreter = tf.lite.Interpreter(model_path=model_path)
        self.interpreter.allocate_tensors()
        self.input_details = self.interpreter.get_input_details()
        self.output_details = self.interpreter.get_output_details()


    def extract_features(self, audio):
        """Extract MFCC with fixed 3-second length"""
        if len(audio) < self.chunk_samples:
            audio = np.pad(audio, (0, self.chunk_samples - len(audio)))
        else:
            audio = audio[:self.chunk_samples]

        mfcc = librosa.feature.mfcc(y=audio, sr=self.sr, n_mfcc=40)
        mfcc = (mfcc - np.mean(mfcc)) / np.std(mfcc)
        return mfcc[..., np.newaxis]


    def predict_chunk(self, audio_chunk):
        features = self.extract_features(audio_chunk)
        features = features[np.newaxis, ...].astype(np.float32)

        self.interpreter.set_tensor(self.input_details[0]['index'], features)
        self.interpreter.invoke()
        pred = self.interpreter.get_tensor(self.output_details[0]['index'])

        return float(pred[0][0])  # probability


    def detect_in_10s(self, file_path):
        """Full pipeline: 10-second audio → chunk predictions"""
        audio, _ = librosa.load(file_path, sr=self.sr)

        chunk_results = []
        elephant_detected = False

        print("\n🔍 Running predictions on chunks...")

        # Split into 3-sec chunks with 1-sec overlap
        step = self.sr * 2  # 1-second overlap
        for start in range(0, len(audio), step):
            end = start + self.chunk_samples
            chunk = audio[start:end]

            if len(chunk) < self.chunk_samples:
                break

            conf = self.predict_chunk(chunk)
            chunk_results.append(conf)

            print(f"Chunk {len(chunk_results)} → Confidence: {conf:.2%}")

            if conf > 0.7:
                elephant_detected = True

        print("\n=== FINAL RESULT ===")
        if elephant_detected:
            return "🐘 Elephant Detected!"
        else:
            return "❌ No Elephant Detected"


In [13]:
if __name__ == "__main__":
    model_path = "elephant_detector.tflite"
    detector = ElephantDetector(model_path)

    # 1. Record
    audio_file = record_audio("recorded_10s.wav", duration=10)

    # 2. Predict using chunk-based detection
    result = detector.detect_in_10s(audio_file)

    print("\n🔥 FINAL OUTPUT:", result)


🎙️ Recording 10 seconds...
✅ Saved recording as recorded_10s.wav

🔍 Running predictions on chunks...
Chunk 1 → Confidence: 0.02%
Chunk 2 → Confidence: 1.02%
Chunk 3 → Confidence: 6.22%
Chunk 4 → Confidence: 3.49%

=== FINAL RESULT ===

🔥 FINAL OUTPUT: ❌ No Elephant Detected
